# 07 — Pronostico

**Librerias usadas:** pandas, numpy, matplotlib, scikit-learn

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'solineras'))
os.chdir(ROOT)
DATA = ROOT / 'data'

p = DATA / 'raw' / 'externas' / 'onei_anuario.csv'
if not p.exists():
    from extraer_onei import extraer
    extraer()
onei = pd.read_csv(p)

## 1. Serie

In [ ]:
s = onei[onei['indicador'] == 'eolica_fotovoltaica_gwh'][['anio','valor']].dropna()
s = s.sort_values('anio').reset_index(drop=True)
s

## 2. Regresion

In [ ]:
if len(s) >= 2:
    X = s['anio'].values.reshape(-1, 1)
    y = s['valor'].values
    model = LinearRegression().fit(X, y)
    anios_fut = np.arange(2025, 2028).reshape(-1, 1)
    pred = model.predict(anios_fut)
    print('Coeficiente:', round(model.coef_[0], 3))
    print('R2:', round(model.score(X, y), 4))
    for a, p_ in zip(anios_fut.flatten(), pred):
        print(f'{a}: {p_:.2f} GWh')

## 3. Grafico

In [ ]:
if len(s) >= 2:
    plt.figure(figsize=(10, 5))
    plt.plot(s['anio'], s['valor'], marker='o', label='Historico')
    plt.plot(anios_fut.flatten(), pred, marker='s', linestyle='--', color='red', label='Pronostico')
    plt.title('Generacion FV + eolica: historico y pronostico')
    plt.xlabel('Anio')
    plt.ylabel('GWh')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

## 4. Metricas

In [ ]:
if len(s) >= 2:
    y_pred = model.predict(X)
    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mape = np.mean(np.abs((y - y_pred) / np.where(y == 0, 1, y))) * 100
    print(f'MAE:  {mae:.2f}')
    print(f'RMSE: {rmse:.2f}')
    print(f'MAPE: {mape:.2f}%')